In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
import importlib
import torch
from lightning.pytorch.callbacks import ModelCheckpoint, RichModelSummary

from rl4co.envs import CVRPEnv
from rl4co.models.zoo.decomposition import DECOMP, DecompPolicy, BaryCenterDecomp, Naive
from rl4co.models.zoo.am import AttentionModel, AttentionModelPolicy
from rl4co.utils.trainer import RL4COTrainer

In [3]:
torch.manual_seed(42)

In [6]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
env = CVRPEnv(generator_params={'num_loc': 200})

In [7]:
policy = AttentionModelPolicy(env_name=env.name, 
                              embed_dim=128,
                              num_encoder_layers=3,
                              num_heads=8,
                            )
naive_pol = Naive()
decomp_pol = BaryCenterDecomp(4, policy=policy, naive_policy=naive_pol).to(device)

model = DECOMP(env,
               policy=decomp_pol,
               baseline='rollout',
               train_data_size=1_000,
               val_data_size=100,
               optimizer_kwargs={"lr": 1e-4})

`None` was provided as encoder. Using `NoEncoder`.
/Users/andrewtao/opt/anaconda3/envs/rl4co/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:199: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/Users/andrewtao/opt/anaconda3/envs/rl4co/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:199: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.


In [14]:
trainer = RL4COTrainer(
    max_epochs=3,
    devices=1,
    logger=None,
)

/Users/andrewtao/opt/anaconda3/envs/rl4co/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/accelerator_connector.py:556: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [16]:
trainer.fit(model)

val_file not set. Generating dataset instead
test_file not set. Generating dataset instead


tensor(-204.4244)


TypeError: 'NoneType' object cannot be interpreted as an integer